# 03 · Training

Trains the DS-CNN phrase spotter on the manifest written by `02_preprocessing`.

Enabled by the config, not by editing this notebook: TensorBoard, mixed precision, early stopping,
checkpointing, resume, class weights, label smoothing, LR schedule, automatic train/val split,
fixed seed, batch size, epochs and optimizer.

**Resuming.** Re-run this notebook with the same `RUN_NAME`; `BackupAndRestore` picks the run up at
the epoch it stopped at, optimizer state included. Set a new `RUN_NAME` to start fresh.

**Runtime → Change runtime type → GPU** before running, or training falls back to CPU.

In [ ]:
#@title Setup — mount Drive, locate the project, install what is missing
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MahmoudMabrok/SaloAleh.git"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project_root() -> Path:
    """The folder that holds src/ and configs/ — cloned if it is not here yet."""
    candidates = [Path(p) for p in [
        os.environ.get("DHIKR_PROJECT_ROOT", ""),
        "/content/DhikrSpeech",
        "/content/SaloAleh/DhikrSpeech",
        "/content/drive/MyDrive/DhikrSpeech",
    ] if p]
    candidates += [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "src" / "config.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            return candidate.resolve()
    if IN_COLAB:
        target = Path("/content/SaloAleh")
        if not target.exists():
            print("cloning", REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        return (target / "DhikrSpeech").resolve()
    raise FileNotFoundError(
        "DhikrSpeech project not found. Set DHIKR_PROJECT_ROOT, or copy the "
        "DhikrSpeech folder to /content or to your Drive."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name, package in [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("yaml", "PyYAML"),
    ("sklearn", "scikit-learn"),
    ("soxr", "soxr"),
]:
    if importlib.util.find_spec(module_name) is None:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
config.paths.ensure_dirs()

print("project root :", PROJECT_ROOT)
print("config       :", CONFIG_PATH)
print()
print(config.summary())


## 1 · Seed, precision, device

In [ ]:
import tensorflow as tf

from src.trainer import configure_mixed_precision, set_global_seed

RUN_NAME = config.model.name  # change to start a separate run

# `training.resume: true` means re-running this notebook restores the previous
# run's weights, optimiser state and epoch counter. That is what you want after a
# Colab disconnect, and exactly what you do not want after changing a
# hyperparameter - the change would be applied on top of the old model and the
# printed history would splice both runs together. Set this to True whenever the
# config changed since the last run.
FRESH_START = False

set_global_seed(config.seed)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed = configure_mixed_precision(config.training.mixed_precision)
print("TensorFlow  :", tf.__version__)
print("GPU         :", [gpu.name for gpu in gpus] or "none — training on CPU")
print("mixed float16:", mixed)
print("run name    :", RUN_NAME)


## 2 · Load the manifest

In [ ]:
import pandas as pd

from src.dataset import class_names_from_manifest, filter_split, load_manifest, split_counts

paths = config.paths
records = load_manifest(paths.manifest_path)
class_names = class_names_from_manifest(records)

train_records = filter_split(records, "train")
val_records = filter_split(records, "val")
test_records = filter_split(records, "test")

if not train_records or not val_records:
    raise ValueError("training needs a non-empty train and val split — re-run 02_preprocessing")

print("classes :", len(class_names), class_names)
print("splits  :", split_counts(records))

display(pd.DataFrame(
    [{"class": record.label, "split": record.split} for record in records]
).pivot_table(index="class", columns="split", aggfunc=len, fill_value=0))

# A split of a few clips per class cannot measure a model, and a handful of clips
# per class cannot train one. Say so here rather than after an hour of training.
val_per_class = len(val_records) / len(class_names)
train_per_class = len(train_records) / len(class_names)
print()
print("train clips / class : %.1f" % train_per_class)
print("val clips / class   : %.1f  (val accuracy moves in steps of %.2f)"
      % (val_per_class, 1.0 / max(len(val_records), 1)))
if val_per_class < 5 or train_per_class < 30:
    print()
    print("!! this dataset is too small to train or to measure a %d-class model."
          % len(class_names))
    print("   Aim for 50-100+ recordings per class from 10+ speakers for a first")
    print("   usable model. Below that, expect the model to collapse to a single")
    print("   class and validation accuracy to sit at chance (%.4f)."
          % (1.0 / len(class_names)))
    print("   The sanity check in section 6b still tells you whether the pipeline")
    print("   itself is correct, which is the useful thing to know meanwhile.")


## 3 · Input pipeline

Training clips are decoded once and cached, then re-augmented every epoch. Validation is never
augmented and never shuffled.

In [ ]:
from src.augmentation import NoiseBank, WaveformAugmentor
from src.dataset import make_tf_dataset
from src.features import FeatureStats, LogMelExtractor

stats_path = paths.processed_path / "feature_stats.json"
stats = FeatureStats.load(stats_path) if (
    config.features.normalize == "global" and stats_path.is_file()
) else None

extractor = LogMelExtractor(config.features, config.audio.sample_rate, stats=stats)
noise_bank = NoiseBank.load(
    paths.noise_path,
    config.audio.sample_rate,
    allow_synthetic=config.augmentation.background_noise.synthetic_when_missing,
)
augmentor = WaveformAugmentor(config.augmentation, config.audio, noise_bank)

train_dataset = make_tf_dataset(train_records, config, extractor, training=True, augmentor=augmentor)
val_dataset = make_tf_dataset(val_records, config, extractor, training=False)

features_batch, labels_batch = next(iter(train_dataset))
print("features:", features_batch.shape, features_batch.dtype)
print("labels  :", labels_batch.shape, labels_batch.dtype)
print("expected:", config.input_shape)
assert tuple(features_batch.shape[1:]) == config.input_shape


## 4 · Class weights

Balanced weights counteract an uneven dataset, so a class with 60 recordings still matters as much
as one with 600. Disable with `training.class_weights: false`.

In [ ]:
from src.dataset import compute_class_weights

class_weight = compute_class_weights(
    [record.class_index for record in train_records], len(class_names)
)
display(pd.DataFrame(
    [
        {
            "class": class_names[index],
            "train clips": sum(1 for record in train_records if record.class_index == index),
            "weight": round(weight, 3),
        }
        for index, weight in sorted(class_weight.items())
    ]
))
print("class weights enabled:", config.training.class_weights)


## 5 · Build the model

DS-CNN: a strided convolutional stem followed by depthwise separable blocks. Small enough for a
phone, and it quantises to INT8 without a Flex delegate.

In [ ]:
from src.models import build_model, estimate_flops, model_summary_text

model = build_model(config.input_shape, len(class_names), config.model)
model.summary()

flops = estimate_flops(model)
print()
print("parameters     : %s" % format(model.count_params(), ","))
print("approx MFLOPs  : %.1f per inference" % (flops / 1e6))
(paths.checkpoints_path / RUN_NAME).mkdir(parents=True, exist_ok=True)
(paths.checkpoints_path / RUN_NAME / "model_summary.txt").write_text(
    model_summary_text(model), encoding="utf-8"
)


## 6 · TensorBoard

Run this before `fit` and the charts update live during training.

In [ ]:
LOG_DIR = paths.logs_path / RUN_NAME / "tensorboard"
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("log dir:", LOG_DIR)

try:
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", "--logdir '%s'" % LOG_DIR)
except Exception as error:  # not running under IPython
    print("start TensorBoard manually:  tensorboard --logdir '%s'" % LOG_DIR)
    print(error)


## 6b · Sanity check — can this pipeline learn at all?

Before spending an hour on a real run, prove that the model can **memorise a handful of clips**.
A few dozen unaugmented recordings, a fresh copy of the model, a couple of hundred steps: training
accuracy has to go to ~1.0. It only has to overfit, so anything less means the fault is upstream of
the hyperparameters — the features, the labels, or the model itself.

This is the test to run first whenever a run sits at chance (0.10 for 10 classes) and the model
predicts one class for everything.


In [ ]:
from src.dataset import make_tf_dataset
from src.trainer import sanity_overfit, sanity_overfit_report

SANITY_CLIPS = 40          # clips to memorise, a few per class
SANITY_STEPS = 200         # optimiser steps to do it in
RUN_SANITY_CHECK = True

if RUN_SANITY_CHECK:
    # Take a stratified handful: at least one clip per class, otherwise the test
    # can pass on a subset that happens to be a single class.
    by_class = {}
    for record in train_records:
        by_class.setdefault(record.class_index, []).append(record)
    subset = []
    position = 0
    while len(subset) < min(SANITY_CLIPS, len(train_records)):
        added = False
        for index in sorted(by_class):
            if position < len(by_class[index]) and len(subset) < SANITY_CLIPS:
                subset.append(by_class[index][position])
                added = True
        if not added:
            break
        position += 1

    # training=False -> no augmentation. Memorising augmented clips is a different,
    # much harder test and not what we are asking here.
    sanity_dataset = make_tf_dataset(
        subset, config, extractor, training=False, batch_size=min(16, len(subset)), shuffle=False
    )
    sanity = sanity_overfit(model, sanity_dataset, steps=SANITY_STEPS)
    print("clips            :", len(subset))
    print(sanity_overfit_report(sanity, len(class_names)))
else:
    print("sanity check skipped")


## 7 · Train

Checkpoints, logs and the config snapshot are written to Drive as training runs, so an interrupted
Colab session loses nothing. Interrupting this cell and re-running it resumes the run.

In [ ]:
import math

from src.trainer import Trainer

steps_per_epoch = math.ceil(len(train_records) / config.training.batch_size)

trainer = Trainer(
    config=config,
    model=model,
    num_classes=len(class_names),
    steps_per_epoch=steps_per_epoch,
    run_name=RUN_NAME,
)

if FRESH_START:
    trainer.reset_run()

trainer.compile()

total_steps = steps_per_epoch * config.training.epochs
print("steps per epoch :", steps_per_epoch)
print("total steps     :", total_steps)
if total_steps < 2000:
    # Convergence follows gradient steps, not epochs. Under a couple of thousand,
    # a DS-CNN trained from scratch is still near its initialisation: the loss
    # falls a little each epoch and accuracy looks pinned near chance.
    print("                  !! under 2000 steps - expect an undertrained model.")
    print("                     Lower training.batch_size or raise training.epochs.")
print("epochs          :", config.training.epochs)
print("checkpoints     :", trainer.checkpoint_dir)
print("resume enabled  :", config.training.resume)
print("resuming a run  :", trainer.is_resuming)
print()

artifacts = trainer.fit(train_dataset, val_dataset, class_weight=class_weight)
print()
print(artifacts.summary())


## 8 · Training curves

In [ ]:
from src import visualization as viz

figure = viz.plot_training_history(artifacts.history, title="run: %s" % RUN_NAME)
viz.save_figure(figure, paths.reports_path / ("03_history_%s.png" % RUN_NAME))

display(pd.DataFrame(artifacts.history).tail(10))


## 9 · Quick check on the validation split

A full evaluation with per-class metrics, confusion matrix and error analysis is
`04_evaluation.ipynb`. This is only a smoke check that the best checkpoint reloads and performs.

In [ ]:
from src.metrics import evaluate_model
from src.trainer import load_trained_model

best_model = load_trained_model(artifacts.best_model_path)
result = evaluate_model(
    best_model,
    val_dataset,
    class_names,
    paths=[record.path for record in val_records],
    confidence_threshold=config.evaluation.confidence_threshold,
)
print(result.summary())
print()
print("predicted class distribution (a healthy model spreads across all classes):")
for label, count in result.prediction_distribution().items():
    print("  %-10s %d" % (label, count))
print()
print("best checkpoint:", artifacts.best_model_path)
print("continue with 04_evaluation.ipynb")
